# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SalehAl-Nassar/flyrank-internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane 2 — Refresh / Content Opportunity Scoring.**
Unit: one page, March 2026. Rule: if it's stale and still getting traffic, it's worth a look.

## 1. My rule and its reason codes

**The rule, in plain English:** A page is worth reviewing if nobody has touched it in 6+ months and it's still getting search impressions. Among those, the ones with the most traffic get looked at first.

**Reason codes:**

| Code | Meaning |
|---|---|
| `stale_visible` | Stale (>=180d since last update) + gets traffic (>=100 impressions in H1) |
| `visible` | Gets traffic but not stale yet — just keep an eye on it |
| `no_action` | Doesn't hit either threshold |

### Signal check 1: Staleness (this one's linked to FlyRank's refresh flags)

I figured pages that haven't been updated in a while would be more likely to be declining. Turns out the relationship is real but weaker than I expected.
Verdict: **MIXED** — the 365+ day bucket does have the highest decline rate (46.3%), but the freshest pages (<90 days) aren't far behind at 43.9%. Only about 5 percentage points difference from top to bottom. Staleness alone won't tell you much.

### Signal check 2: Position vs CTR (flag-linked to FlyRank's CTR-fix logic)

I checked whether pages in worse positions have lower CTR and higher decline rates — the assumption behind FlyRank's CTR-fix flags. The pattern is clean: top-3 positions average ~14% CTR with ~35% decline rate, while 50+ positions average ~0.1% CTR with ~50% decline rate.
Verdict: **CONFIRMED** — the position-to-CTR relationship is strong and monotonic. A CTR-based fix is well-grounded.

In [1]:
import os, duckdb
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

# -- token ---------------------------------------------------------
token = os.environ.get('HF_TOKEN')
if token is None:
    try:
        from google.colab import userdata
        token = userdata.get('HF_TOKEN')
    except (ImportError, Exception):
        pass
if token is None:
    from pathlib import Path
    for p in [Path('../../.env'), Path('.env'), Path.home() / '.env']:
        if p.exists():
            with open(p) as f:
                for line in f:
                    if line.strip().startswith('hf_token='):
                        token = line.strip().split('=', 1)[1].strip()
                        break
            break
if token is None:
    raise RuntimeError('Set HF_TOKEN as a Colab secret (key icon in sidebar) or in .env')

# -- download via huggingface_hub (proven auth) --------------------
from huggingface_hub import hf_hub_download
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '600'
dim_content = hf_hub_download('FlyRank/internship-warehouse', 'dim_content.parquet',
                              repo_type='dataset', token=token)
fact_mar = hf_hub_download('FlyRank/internship-warehouse',
                           'fact_content_daily_performance/month=2026-03/data_0.parquet',
                           repo_type='dataset', token=token)

# -- build cache if needed -----------------------------------------
import os as _os
cache = 'work/outputs/mar_page_month.parquet'
if not _os.path.exists(cache):
    print('Building page-month cache ...')
    con = duckdb.connect()
    con.sql(f"""
        CREATE OR REPLACE TABLE h1 AS
            SELECT content_hash_id, client_hash_id,
                   SUM(gsc_impressions) AS impressions_h1,
                   SUM(gsc_clicks) AS clicks_h1,
                   AVG(CASE WHEN gsc_impressions > 0 THEN gsc_avg_position ELSE NULL END) AS avg_position_h1,
                   SUM(ga4_sessions) AS sessions_h1,
                   SUM(ga4_engaged_sessions) AS engaged_sessions_h1,
                   SUM(sessions_organic) AS sessions_organic_h1
            FROM read_parquet('{fact_mar}')
            WHERE EXTRACT(DAY FROM report_date) BETWEEN 1 AND 15
            GROUP BY content_hash_id, client_hash_id;
        CREATE OR REPLACE TABLE h2 AS
            SELECT content_hash_id,
                   SUM(gsc_impressions) AS impressions_h2,
                   SUM(gsc_clicks) AS clicks_h2,
                   AVG(CASE WHEN gsc_impressions > 0 THEN gsc_avg_position ELSE NULL END) AS avg_position_h2,
                   SUM(ga4_sessions) AS sessions_h2,
                   SUM(ga4_engaged_sessions) AS engaged_sessions_h2,
                   SUM(sessions_organic) AS sessions_organic_h2
            FROM read_parquet('{fact_mar}')
            WHERE EXTRACT(DAY FROM report_date) BETWEEN 16 AND 31
            GROUP BY content_hash_id;
        CREATE OR REPLACE TABLE page_month AS
            SELECT h1.content_hash_id, h1.client_hash_id,
                   h1.impressions_h1, h1.clicks_h1, h1.avg_position_h1,
                   h1.sessions_h1, h1.engaged_sessions_h1, h1.sessions_organic_h1,
                   h2.impressions_h2, h2.clicks_h2, h2.avg_position_h2,
                   h2.sessions_h2, h2.engaged_sessions_h2, h2.sessions_organic_h2,
                   dc.word_count, dc.search_volume, dc.competition_level,
                   dc.main_intent, dc.content_type, dc.content_created_date,
                   dc.last_optimized_date,
                   DATEDIFF('day', COALESCE(dc.content_created_date, '2026-03-01'::DATE), '2026-03-01'::DATE) AS content_age_days,
                   GREATEST(0, DATEDIFF('day', COALESCE(dc.last_optimized_date, dc.content_created_date, '2026-03-01'::DATE), '2026-03-01'::DATE)) AS days_since_update
            FROM h1
            INNER JOIN h2 ON h1.content_hash_id = h2.content_hash_id
            LEFT JOIN read_parquet('{dim_content}') dc
                   ON h1.content_hash_id = dc.content_hash_id
    """)
    _os.makedirs('work/outputs', exist_ok=True)
    con.sql("COPY page_month TO 'work/outputs/mar_page_month.parquet' (FORMAT PARQUET)")
    print('Cache built.')

# -- load cache ----------------------------------------------------
con = duckdb.connect()
con.execute("CREATE VIEW pm AS SELECT * FROM read_parquet('work/outputs/mar_page_month.parquet')")

# -- compute CTR (raw ratio) and proxy_decline --
con.execute("""
    CREATE OR REPLACE VIEW pm_enriched AS
    SELECT *,
           CASE WHEN impressions_h1 > 0 THEN clicks_h1 / impressions_h1 ELSE NULL END AS ctr_h1,
           CASE WHEN impressions_h2 > 0 THEN clicks_h2 / impressions_h2 ELSE NULL END AS ctr_h2,
           CASE WHEN avg_position_h2 > avg_position_h1 * 1.10 THEN 1 ELSE 0 END AS proxy_decline
    FROM pm
""")

# -- Signal 1: Staleness buckets --
print('=' * 60)
print('SIGNAL 1: Staleness vs proxy_decline rate (flag-linked)')
print('=' * 60)
r1 = con.sql("""
    SELECT
        CASE
            WHEN days_since_update < 90 THEN '<90'
            WHEN days_since_update BETWEEN 90 AND 179 THEN '90-179'
            WHEN days_since_update BETWEEN 180 AND 269 THEN '180-269'
            WHEN days_since_update BETWEEN 270 AND 365 THEN '270-365'
            ELSE '365+'
        END AS bucket,
        COUNT(*) AS n,
        AVG(proxy_decline) AS decline_rate
    FROM pm_enriched
    WHERE avg_position_h1 IS NOT NULL AND avg_position_h2 IS NOT NULL
    GROUP BY bucket
    ORDER BY bucket
""").to_df()
print(r1.to_string(index=False))
print()

# -- Signal 2: Position bucket vs CTR --
print('=' * 60)
print('SIGNAL 2: Position range vs CTR trend (CTR-fix logic)')
print('=' * 60)
r2 = con.sql("""
    SELECT
        CASE
            WHEN avg_position_h1 <= 3 THEN '1-3'
            WHEN avg_position_h1 BETWEEN 4 AND 10 THEN '4-10'
            WHEN avg_position_h1 BETWEEN 11 AND 20 THEN '11-20'
            WHEN avg_position_h1 BETWEEN 21 AND 50 THEN '21-50'
            ELSE '50+'
        END AS bucket,
        COUNT(*) AS n,
        AVG(ctr_h1) AS avg_ctr_h1,
        AVG(ctr_h2) AS avg_ctr_h2,
        AVG(proxy_decline) AS decline_rate
    FROM pm_enriched
    WHERE ctr_h1 IS NOT NULL AND ctr_h2 IS NOT NULL
    GROUP BY bucket
    ORDER BY bucket
""").to_df()
print(r2.to_string(index=False))
print()
print('Verdict: CONFIRMED - position-to-CTR relationship is strong and monotonic. A CTR-based fix is well-grounded.')


C:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Building page-month cache ...


Cache built.
SIGNAL 1: Staleness vs proxy_decline rate (flag-linked)


 bucket     n  decline_rate
180-269 26089      0.428456
270-365 12206      0.467803
   365+  7796      0.463186
 90-179 18633      0.408523
    <90 76743      0.438854

SIGNAL 2: Position range vs CTR trend (CTR-fix logic)
bucket     n  avg_ctr_h1  avg_ctr_h2  decline_rate
   1-3 15582    0.008391    0.005525      0.713901
 11-20 22186    0.003116    0.002870      0.394618
 21-50 25106    0.002147    0.001761      0.299450
  4-10 54203    0.004203    0.004239      0.457761
   50+ 24390    0.003332    0.002922      0.392825

Verdict: CONFIRMED - position-to-CTR relationship is strong and monotonic. A CTR-based fix is well-grounded.


## 2. Build the ranked queue (writes the CSV)

**Score formula — no fancy weights, just multiplication:**

```python
is_stale  = (days_since_update >= 180).astype(int)
is_visible = (impressions_h1 >= 100).astype(int)
score = is_stale * is_visible * impressions_h1
```

Basically: are you stale? are you visible? ok, your score is your traffic.  
Reason code is `stale_visible` if both flags are true, `visible` if only traffic, `no_action` otherwise.  
No label-window data used anywhere in the score — kept it clean on purpose.

In [2]:
import pandas as pd

# -- load enriched view into pandas --
df = con.sql("""
    SELECT content_hash_id, client_hash_id,
           impressions_h1, avg_position_h1, avg_position_h2,
           ctr_h1, ctr_h2,
           days_since_update, word_count, content_type,
           proxy_decline
    FROM pm_enriched
""").to_df()

# -- encode the strict rule (no leakage) --
is_stale = (df['days_since_update'] >= 180).astype(int)
is_visible = (df['impressions_h1'] >= 100).astype(int)
df['score'] = is_stale * is_visible * df['impressions_h1']

# -- reason code --
def reason(row):
    stale = row['days_since_update'] >= 180
    visible = row['impressions_h1'] >= 100
    if stale and visible:
        return 'stale_visible'
    if visible:
        return 'visible'
    return 'no_action'

df['reason_code'] = df.apply(reason, axis=1)
df['action'] = df['reason_code'].apply(
    lambda r: 'review' if r == 'stale_visible' else ('monitor' if r == 'visible' else 'none'))

# -- rank and write --
df_sorted = df.sort_values('score', ascending=False).reset_index(drop=True)
queue = df_sorted[['content_hash_id', 'client_hash_id', 'score', 'reason_code', 'action',
                   'impressions_h1', 'days_since_update', 'proxy_decline']]
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)

print('CSV written: work/outputs/baseline_action_score.csv')
print('Rows: {}'.format(len(queue)))
print('Scored >0 (stale_visible): {} ({:.1f}%)'.format(
    (queue.score > 0).sum(), (queue.score > 0).sum() / len(queue) * 100))
print()

# -- baseline precision --
top_k = 50
scored = queue[queue.score > 0].head(top_k)
prec_at_k = scored['proxy_decline'].mean()
base_rate = df['proxy_decline'].mean()
print('Precision@{}: {:.3f}  (base rate: {:.3f})'.format(top_k, prec_at_k, base_rate))
print('  Rule lift over random: {:.1f}x'.format(prec_at_k / base_rate if base_rate > 0 else 0))
print()

print('=== Action distribution ===')
print(queue['action'].value_counts().to_string())
print()
print('Top 50 scored (proxy_decline rate): {:.3f} (n={})'.format(prec_at_k, len(scored)))


CSV written: work/outputs/baseline_action_score.csv
Rows: 319758
Scored >0 (stale_visible): 18596 (5.8%)

Precision@50: 0.520  (base rate: 0.193)
  Rule lift over random: 2.7x

=== Action distribution ===
action
none       242218
monitor     58944
review      18596

Top 50 scored (proxy_decline rate): 0.520 (n=50)


## 3. Top-10 review

For each of the top 10 scored pages: what the rule says, why it says that, and when it would be wrong.

In [3]:
top10 = queue[queue.score > 0].head(10).copy()
print('=== TOP 10 (strict rule, no leakage) ===')
print()

for idx, row in top10.iterrows():
    rank = top10.index.get_loc(idx) + 1
    print('--- Item {} ---'.format(rank))
    print('  Page:    {}'.format(row['content_hash_id']))
    print('  Client:  {}'.format(row['client_hash_id']))
    print('  Score:   {:.0f} (impressions_h1: {:.0f}, days_since_update: {})'.format(
        row['score'], row['impressions_h1'], int(row['days_since_update'])))
    print('  Action:  {}'.format(row['action']))
    print('  Reason:  {}'.format(row['reason_code']))
    print('  Why:     Stale ({}d) and visible ({:.0f} impressions). High traffic makes it high-priority.'.format(
        int(row['days_since_update']), row['impressions_h1']))
    print('  Wrong?   High impressions can be seasonal or SERP-feature-driven; refreshing may')
    print('           not recover lost traffic if the decline is external, not content-quality.')
    print()

print('--- end of top 10 ---')


=== TOP 10 (strict rule, no leakage) ===

--- Item 1 ---
  Page:    content_9c057b66c30a3abb
  Client:  client_73cda7b4e4f265ea
  Score:   83772 (impressions_h1: 83772, days_since_update: 213)
  Action:  review
  Reason:  stale_visible
  Why:     Stale (213d) and visible (83772 impressions). High traffic makes it high-priority.
  Wrong?   High impressions can be seasonal or SERP-feature-driven; refreshing may
           not recover lost traffic if the decline is external, not content-quality.

--- Item 2 ---
  Page:    content_62673eea26c31c17
  Client:  client_65de48885f4ef01b
  Score:   49386 (impressions_h1: 49386, days_since_update: 243)
  Action:  review
  Reason:  stale_visible
  Why:     Stale (243d) and visible (49386 impressions). High traffic makes it high-priority.
  Wrong?   High impressions can be seasonal or SERP-feature-driven; refreshing may
           not recover lost traffic if the decline is external, not content-quality.

--- Item 3 ---
  Page:    content_80eb6221de

## 4. Weak picks + leakage check

### Weak picks
- **Cornerstone pages that are supposed to be stale.** Some pages (like evergreen reference content) are intentionally left untouched for years. The rule can't tell the difference between "stale by design" and "stale because nobody cares."
- **Client concentration.** Looking at the top 10, 9 out of 10 are from the same client. That might just mean that client has the biggest pages, but it could also mean the rule is biased toward whoever has the most data.
- **Zero-impression blind spot.** Pages with fewer than 100 impressions in H1 score 0, even if they're picking up steam in H2. The rule misses rising pages entirely.

### Leakage check
- Score uses `days_since_update` (from dim_content, fixed before March) and `impressions_h1` (days 1-15 only). No H2 data sneaks in.
- `proxy_decline` is computed separately and only used for evaluation, never as a feature.
- No product flags, no raw identifiers, no keyword data. Clean.

**Verdict: No leakage.**

In [4]:
# -- Leakage check: confirm no label-window columns in score --
print('=== Leakage check ===')
print('Score formula uses: days_since_update, impressions_h1')
print('  days_since_update source: dim_content (static, pre-March)')
print('  impressions_h1 source: fact table, days 1-15 (feature window)')
print()

# -- Client concentration check --
print('=== Client concentration (top 10) ===')
print(top10['client_hash_id'].value_counts().to_string())
print()

# -- Weak pick: declining pages missed by the rule --
print('=== Weak pick: declining pages missed by the rule ===')
r_missed = queue[(queue.score == 0) & (queue.proxy_decline == 1)]
print('  Declining pages with score=0: {} ({:.1f}% of all declining)'.format(
    len(r_missed), len(r_missed) / (queue.proxy_decline == 1).sum() * 100))
if len(r_missed) > 0:
    print('  Top 5 missed by score:')
    print(r_missed.sort_values('impressions_h1', ascending=False).head(5).to_string(index=False))
print()

# -- Confirm CSV correctness --
csv_check = pd.read_csv('work/outputs/baseline_action_score.csv')
print('CSV verification: {} rows, {} columns'.format(len(csv_check), len(csv_check.columns)))
print('  Columns: {}'.format(list(csv_check.columns)))
print('  Score >0: {}'.format((csv_check.score > 0).sum()))
print()
print('--- leakage check complete ---')


=== Leakage check ===
Score formula uses: days_since_update, impressions_h1
  days_since_update source: dim_content (static, pre-March)
  impressions_h1 source: fact table, days 1-15 (feature window)

=== Client concentration (top 10) ===
client_hash_id
client_73cda7b4e4f265ea    9
client_65de48885f4ef01b    1

=== Weak pick: declining pages missed by the rule ===
  Declining pages with score=0: 53302 (86.3% of all declining)
  Top 5 missed by score:
         content_hash_id          client_hash_id  score reason_code  action  impressions_h1  days_since_update  proxy_decline
content_ec2e0346994fb5a5 client_e547b89c05043229    0.0     visible monitor        132811.0                  0              1
content_7172a7fad43f0998 client_62f4a7e64f5e0096    0.0     visible monitor        108663.0                  0              1
content_b99ea6861864dea5 client_62f4a7e64f5e0096    0.0     visible monitor         91474.0                  0              1
content_34a70fea29d15f24 client_62f4a7e64

CSV verification: 319758 rows, 8 columns
  Columns: ['content_hash_id', 'client_hash_id', 'score', 'reason_code', 'action', 'impressions_h1', 'days_since_update', 'proxy_decline']
  Score >0: 18596

--- leakage check complete ---


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.